# Transformers & Large Language Models
## Auxiliary Lectures for EEE 5xx & 6xx Courses

### By: David Ramirez ([GitHub](https://github.com/dframirez-usmc))

In [ ]:
# Check for Nvidia CUDA (Nvidia CUDA Compiler)
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [ ]:
# Check for Nvidia GPU (System Management Interface)
!nvidia-smi

Wed Nov 19 19:33:32 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#!pip install transformers accelerate --upgrade #--quiet

In [ ]:
# This pip install assumes you have CUDA 12.4 installed
# CUDA setup is usually seperate from python or pip
#!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 --upgrade #--quiet

In [ ]:
# It may be necessary to uninstall in order to reinstall PyTorch
#!pip uninstall torch torchvision torchaudio --yes

In [ ]:
# Check against your installed pip packages
!pip list

Package                                  Version
---------------------------------------- --------------------
absl-py                                  1.4.0
absolufy-imports                         0.3.1
accelerate                               1.11.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.2
aiosignal                                1.4.0
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.17.1
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
antlr4-python3-runtime                   4.9.3
anyio                                    4.11.0
anywidget                                0.9.19
argon2-cffi                        

In [ ]:
import os
import torch
import numpy as np

In [ ]:
torch.cuda.is_available()

True

In [ ]:
torch.cuda.get_device_name()

'Tesla T4'

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "openai-community/openai-gpt"

tokenizer = AutoTokenizer.from_pretrained(model_id)
print("tokenizer.vocab_size =")
print(tokenizer.vocab_size)
print("^ Tokenizer vocabulary size", end="\n\n")

model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16).to("cuda")

input_text = "The ASU Fulton Schools of Engineering are"
print("input_text =")
print(input_text)
print("^ Text to use as input", end="\n\n")

input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda")
print("input_ids =")
print(input_ids.cpu().numpy()[0])
print("^ Input text token IDs", end="\n\n")

outputs = model.generate(
    input_ids,
    max_new_tokens=16,
    do_sample=True,
    return_dict_in_generate=True,
    output_logits=True,
    )

logits = outputs.logits
print("len(logits) =")
print(len(logits))
print("^ Quantity of new generated token logits", end="\n\n")

logit_vector = outputs.logits[0][0]
print("len(logit_vector) =")
print(len(logit_vector.cpu().numpy()))
print("^ Token logits vector size", end="\n\n")

print("logit_vector =")
print(logit_vector.cpu().numpy()[0:3], end="")
print(" ... ", end="")
print(logit_vector.cpu().numpy()[-4:-1])
print("^ Token logit raw values", end="\n\n")

softmax = torch.nn.functional.softmax(logit_vector, dim=-1)
percents = [f'{i*100:.8f}%' for i in softmax.cpu().numpy()]
print("softmax(logit_vector) =")
print(percents[0:3], end="")
print(" ... ", end="")
print(percents[-4:-1])
print("^ Softmax of logit values", end="\n\n")

print("argmax(softmax) =")
print(torch.argmax(softmax).cpu().numpy())
print("max(softmax) =")
max_percent = '{:.2%}'.format(torch.max(softmax).cpu().numpy())
print(max_percent)
print("^ Most probably next token IDs and value", end="\n\n")

print("outputs =")
print(outputs[0][0].cpu().numpy())
print("^ All output token IDs", end="\n\n")

new_outputs = outputs[0][0][9:].cpu().numpy()
print("new_outputs =")
print(new_outputs)
print("^ Only next output token IDs", end="\n\n")

print("tokenizer.decode(new_outputs) =")
print(tokenizer.decode(new_outputs, skip_special_tokens=True))
print("^ Next token output text", end="\n\n")

print("tokenizer.decode(outputs) =")
generated_text = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
# Format the code output
formatted_code = generated_text.strip()
print(formatted_code)
print("^ All the output text", end="\n\n")

tokenizer.vocab_size =
40478
^ Tokenizer vocabulary size

input_text =
The ASU Fulton Schools of Engineering are
^ Text to use as input

input_ids =
[  481   621   254  4738  1645 12207   498 14283   640]
^ Input text token IDs

len(logits) =
16
^ Quantity of new generated token logits

len(logit_vector) =
40478
^ Token logits vector size

logit_vector =
[ -8.6328125  -5.2695312 -15.6171875] ... [-13.984375 -12.3125   -15.125   ]
^ Token logit raw values

softmax(logit_vector) =
['0.00004186%', '0.00120917%', '0.00000004%'] ... ['0.00000020%', '0.00000106%', '0.00000006%']
^ Softmax of logit values

argmax(softmax) =
481
max(softmax) =
4.12%
^ Most probably next token IDs and value

outputs =
[  481   621   254  4738  1645 12207   498 14283   640   620  7971   525
  1099  5146   498   481  5864   956   544   668  6024   485  1456   488
  4235]
^ All output token IDs

new_outputs =
[ 620 7971  525 1099 5146  498  481 5864  956  544  668 6024  485 1456
  488 4235]
^ Only next output toke